# 🏥 Realistic Emergency Room Simulation using Gymnasium
This notebook defines a custom Gymnasium environment that simulates a realistic emergency room.
We model different types of patients (mild, moderate, critical), patient inflow, treatment delay effects,
and implement a simple policy agent to act within the environment.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random

In [ ]:
class RealisticEmergencyRoomEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(4)  # 0 = treat mild, 1 = moderate, 2 = critical, 3 = idle
        self.observation_space = spaces.Box(low=0, high=100, shape=(4,), dtype=np.int32)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.patients = {"mild": 5, "moderate": 3, "critical": 2}
        self.critical_wait_time = 0
        self.time = 0
        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([
            self.patients["mild"],
            self.patients["moderate"],
            self.patients["critical"],
            self.time
        ], dtype=np.int32)

    def step(self, action):
        reward = 0
        self.time += 1
        self.critical_wait_time += 1
        if action == 0 and self.patients["mild"] > 0:
            self.patients["mild"] -= 1
            reward = 2
        elif action == 1 and self.patients["moderate"] > 0:
            self.patients["moderate"] -= 1
            reward = 5
        elif action == 2 and self.patients["critical"] > 0:
            self.patients["critical"] -= 1
            reward = 20
            self.critical_wait_time = 0
        elif action == 3:
            reward = -1
        else:
            reward = -5
        if self.critical_wait_time >= 3 and self.patients["critical"] > 0:
            self.patients["critical"] -= 1
            reward -= 20
            self.critical_wait_time = 0
        if self.time % 2 == 0:
            new_case = random.choices(["mild", "moderate", "critical"], weights=[0.5, 0.3, 0.2])[0]
            self.patients[new_case] += 1
        done = self.time >= 50
        return self._get_obs(), reward, done, False, {}

    def render(self):
        print(f"Time: {self.time} | Mild: {self.patients['mild']} | Moderate: {self.patients['moderate']} | Critical: {self.patients['critical']}")

In [ ]:
env = RealisticEmergencyRoomEnv()
obs, _ = env.reset()
env.render()

In [ ]:
# Sumulate delays and actions
for _ in range(50):
    action = env.action_space.sample()  # Random action for demonstration
    obs, reward, done, _, _ = env.step(action)
    env.render()
    if done:
        print("Simulation ended.")
        break
    print(f"Action taken: {action}, Reward: {reward}")
    print("-" * 40)
    if done:
        break
    print("End of episode.")
    print("Simulation complete.")
    print("Final state:")
    env.render()
    print("Thank you for using the Realistic Emergency Room Environment!")


In [ ]:
# importing sleep to simulate time passing
from time import sleep

In [ ]:
obs, _ = env.reset()
done = False
total_reward = 0
while not done:
    mild, moderate, critical, time = obs
    if critical > 0:
        action = 2  # treat critical
    elif moderate > 0:
        action = 1  # treat moderate
    elif mild > 0:
        action = 0  # treat mild
    else:
        action = 3  # idle
    obs, reward, done, _, _ = env.step(action)
    env.render()
    total_reward += reward
    sleep(0.5)  # simulate time passing

print("Total reward:", total_reward)

## 🎓 Exercise: Solve the Emergency Room Environment using Monte Carlo Methods

Your task is to implement and compare the performance of two Monte Carlo prediction methods:

**1. First-Visit Monte Carlo (MC)**
- Estimate the value of states by averaging returns **only from the first time** a state is visited in each episode.

**2. Every-Visit Monte Carlo (MC)**
- Estimate the value of states by averaging **all returns** for each state, including repeated visits within the same episode.

**Steps:**
1. Generate episodes using a random policy.
2. Implement both algorithms separately.
3. Track the estimated value of key states (e.g., `[10, 1]`, `[5, 1]`, `[0, 1]`).
4. Plot value estimates vs episodes.

👉 Use a discount factor `γ = 0.9`.

This will help you observe how the two MC approaches converge and how the initial state values evolve over time.
